# PHEE MVP Evaluation
This notebook evaluates the LLM extractions for all 5 splits using the official PHEE metrics.
It replaces the `run_evaluation` script from the `.py` file.


In [18]:
import os
import sys
import json
from collections import defaultdict

# Add project paths to find modules
# Since the script is in 'evaluate/', the project root is '..'
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from helper import (
    load_data, parse_llm_output, GOLD_PATH_TEMPLATE, PHEE_METRIC_DIR, 
    EVENT_TYPE_MAP, ARG_TYPE_MAP, MISSING_VALUES, LLM_DICT
)
from evaluate import read_gold_result, map_predictions, print_phee_results


In [19]:
import math

def calculate_entropy(token_logprobs):
    """
    Berechnet die durchschnittliche Entropie über alle Tokens einer Vorhersage.
    """
    total_entropy = 0
    if not token_logprobs:
        return float('inf')
        
    for token_pos in token_logprobs:
        pos_entropy = 0
        for entry in token_pos:
            logp = entry['logprob']
            if logp == "-inf":
                continue
            p = math.exp(float(logp))
            pos_entropy -= p * float(logp)
        total_entropy += pos_entropy
        
    return total_entropy / len(token_logprobs)

def majority_voting_entropy(all_permutation_json_results, all_permutations, n_voting=5):
    """
    Führt Majority Voting basierend auf Entropie durch.
    Für jede Testinstanz werden die 'n_voting' Permutationen mit der niedrigsten
    durchschnittlichen Entropie ausgewählt und deren Ergebnis aggregiert.
    """
    # all_permutation_json_results ist eine Liste von Listen (pro Permutation eine Liste von Result-Dicts)
    n_instances = len(all_permutation_json_results[0])
    aggregated_preds = []
    
    for i in range(n_instances):
        # Bestimme Entropie für diese Instanz über alle verfügbaren Permutationen
        instance_variants = []
        for p_idx, perm in enumerate(all_permutations):
            res_data = all_permutation_json_results[p_idx][i]
            entropy = calculate_entropy(res_data.get('token_logprobs', []))
            
            # Normalisierung des Texts
            pred_str = res_data['text']
            # WICHTIG: parse_llm_output muss korrekt funktionieren (mit dem leading "[" Fix in helper.py)
            parsed_tuples = parse_llm_output(pred_str)
            
            field_to_idx = {field: idx for idx, field in enumerate(perm)}
            norm_indices = [field_to_idx[f] for f in ORDER_MAIN]
            
            norm_tuples = []
            for t in parsed_tuples:
                if len(t) == len(ORDER_MAIN):
                    norm_tuples.append(tuple(t[idx] for idx in norm_indices))
            
            instance_variants.append({
                'entropy': entropy,
                'tuples': norm_tuples
            })
            
        # Sortiere nach niedrigster Entropie und nimm Top N
        instance_variants.sort(key=lambda x: x['entropy'])
        top_n = instance_variants[:n_voting]
        
        # Majority Voting über die Top N
        tuple_counts = defaultdict(int)
        for variant in top_n:
            for t in variant['tuples']:
                tuple_counts[t] += 1
        
        # Threshold für Majority (n/2 + 1)
        threshold = (len(top_n) // 2) + 1
        final_tuples = [t for t, count in tuple_counts.items() if count >= threshold]
        aggregated_preds.append(str(final_tuples))
        
    return aggregated_preds


In [20]:
def majority_voting(all_permutation_preds, all_permutations, majority_threshold=None):
    """
    Aggrigiert die Ergebnisse der verfügbaren Permutationen mittels Majority Voting.
    Falls kein majority_threshold angegeben ist, wird die absolute Mehrheit 
    der vorhandenen Listen (len/2 + 1) verwendet.
    """
    n_instances = len(all_permutation_preds[0])
    n_perms = len(all_permutation_preds)
    
    if majority_threshold is None:
        majority_threshold = (n_perms // 2) + 1
        
    print(f"Using majority threshold: {majority_threshold} (from {n_perms} available permutations)")
    
    aggregated_preds = []
    
    for i in range(n_instances):
        tuple_counts = defaultdict(int)
        
        for p_idx, perm in enumerate(all_permutations):
            pred_str = all_permutation_preds[p_idx][i]
            parsed_tuples = parse_llm_output(pred_str)
            
            # Normalisierung auf ORDER_MAIN
            field_to_idx = {field: idx for idx, field in enumerate(perm)}
            norm_indices = [field_to_idx[f] for f in ORDER_MAIN]
            
            for t in parsed_tuples:
                if len(t) == len(ORDER_MAIN):
                    norm_t = tuple(t[idx] for idx in norm_indices)
                    tuple_counts[norm_t] += 1
        
        # Behalte nur Tuples mit Mehrheit
        final_tuples = [t for t, count in tuple_counts.items() if count >= majority_threshold]
        aggregated_preds.append(str(final_tuples))
        
    return aggregated_preds

In [21]:
import itertools
from helper import ORDER_MAIN, LLM_DICT

# Configuration
SPLITS = [2]
N_SHOTS = 100
N_VOTING = 5
MODEL_NAME = "google/gemma-4-31B"
RESULTS_DIR = os.path.join(project_root, "results")
OVERALL_STATS = {}

# Get the file-friendly model name
model_file_path = LLM_DICT[MODEL_NAME]["file_path"]

for split in SPLITS:
    print(f"\n{'='*25} EVALUATING SPLIT {split} {'='*25}")
    gold_file = GOLD_PATH_TEMPLATE.format(split=split)
    
    if not os.path.exists(gold_file):
        print(f"Skipping split {split}: Gold file not found.")
        continue
        
    gold_instances = read_gold_result(gold_file)
    
    # Collect all available permutation results (JSON for entropy/preds)
    all_permutations = list(itertools.permutations(ORDER_MAIN))
    all_perm_json_results = []
    valid_perms = []
    
    for p_idx, perm in enumerate(all_permutations):
        # Specific naming pattern: split{split}_shots{n_shots}_{model_file_path}_main_mvp_p{p_idx}
        filename = f"split{split}_shots{N_SHOTS}_{model_file_path}_main_mvp_p{p_idx}.json"
        json_path = os.path.join(RESULTS_DIR, filename)
        
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                all_perm_json_results.append(data['results'])
                valid_perms.append(perm)

    # 1. Entropy-based Majority Voting Results
    res_entropy_mvp = None
    if len(all_perm_json_results) > 0:
        actual_n = min(len(all_perm_json_results), N_VOTING)
        print(f"Loaded {len(all_perm_json_results)} permutations. Calculating Top {actual_n} Voting...")
        entropy_aggregated = majority_voting_entropy(all_perm_json_results, valid_perms, n_voting=N_VOTING)
        res_entropy_mvp = map_predictions(entropy_aggregated, gold_instances, format_type='main')
        OVERALL_STATS[f"split{split}_main_mvp_entropy_maj"] = res_entropy_mvp
    else:
        print(f"Skipping MVP Majority: No result files found in {RESULTS_DIR}")

    # 2. Extract "Single" result from the first permutation (ORDER_MAIN)
    res_single = None
    try:
        found_single_idx = -1
        for i, p in enumerate(valid_perms):
            if p == tuple(ORDER_MAIN):
                found_single_idx = i
                break
        
        if found_single_idx != -1:
            single_preds = [r['text'] for r in all_perm_json_results[found_single_idx]]
            res_single = map_predictions(single_preds, gold_instances, format_type='main')
            OVERALL_STATS[f"split{split}_main_single"] = res_single
    except Exception as e:
        print(f"Error extracting single result: {e}")

    # Print Comparison Table
    print(f"\nSummary for Split {split} ({MODEL_NAME}):")
    print(f"{'Mode':<35} | {'EM F1':<10} | {'Token F1':<10}")
    print("-" * 61)
    
    if res_single:
        print(f"{'Main_Single (Standard)':<35} | {res_single.get('CLS_MainArgs_EM_F1', 0):>10.2f} | {res_single.get('CLS_MainArgs_MICRO_F1', 0):>10.2f}")
    if res_entropy_mvp:
        mode_label = f"Main_MVP_Entropy_Maj (Top {actual_n})"
        print(f"{mode_label:<35} | {res_entropy_mvp.get('CLS_MainArgs_EM_F1', 0):>10.2f} | {res_entropy_mvp.get('CLS_MainArgs_MICRO_F1', 0):>10.2f}")

# Save final summary
summary_filename = f"summary_{model_file_path}_shots{N_SHOTS}.json"
summary_path = os.path.join(RESULTS_DIR, summary_filename)
with open(summary_path, "w") as f:
    json.dump(OVERALL_STATS, f, indent=4)
print(f"\nFinal summary saved to {summary_path}")



========================= EVALUATING SPLIT 2 =========================
Loaded 24 permutations. Calculating Top 5 Voting...

Summary for Split 2 (google/gemma-4-31B):
Mode                                | EM F1      | Token F1  
-------------------------------------------------------------
Main_Single (Standard)              |      66.68 |      76.44
Main_MVP_Entropy_Maj (Top 5)        |      66.09 |      76.73

Final summary saved to /home/hellwig/medical/phee-mvp/results/summary_google_gemma-4-31B_shots100.json
